# Day 027 Project Solution — DailyBriefingScheduler

A `DailyBriefingScheduler` that generates AI briefings on an interval schedule with idempotent run tracking.

In [ ]:
import json, os, tempfile
from datetime import datetime
from pathlib import Path
import ollama


def build_job(
    name: str,
    fn,
    interval_minutes: int,
    enabled: bool = True,
) -> dict:
    if interval_minutes <= 0:
        raise ValueError(f"interval_minutes must be > 0, got {interval_minutes}")
    return {
        "name": name,
        "fn_name": fn.__name__,
        "interval_minutes": interval_minutes,
        "enabled": enabled,
    }


def is_due(
    last_run_iso: str | None,
    interval_minutes: int,
    now: datetime | None = None,
) -> bool:
    if last_run_iso is None:
        return True
    if now is None:
        now = datetime.now()
    last_run = datetime.fromisoformat(last_run_iso)
    elapsed_seconds = (now - last_run).total_seconds()
    return elapsed_seconds >= interval_minutes * 60


def save_run_log(path: str, records: list[dict]) -> None:
    Path(path).write_text(json.dumps(records, indent=2), encoding="utf-8")


def load_run_log(path: str) -> list[dict]:
    p = Path(path)
    if not p.exists():
        return []
    return json.loads(p.read_text(encoding="utf-8"))


def record_run(
    log: list[dict],
    name: str,
    result: str,
    status: str = "ok",
) -> list[dict]:
    new_record = {
        "name": name,
        "ran_at": datetime.now().isoformat(),
        "result": result,
        "status": status,
    }
    return log + [new_record]


def ai_daily_briefing(topics: list[str], model: str = "llama3.2") -> str:
    topics_str = "\n".join(f"- {t}" for t in topics)
    response = ollama.chat(
        model=model,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a professional daily briefing assistant. "
                    "Write a concise, structured briefing covering the given topics. "
                    "Use clear section headers (## Topic). Keep it under 300 words."
                ),
            },
            {
                "role": "user",
                "content": (
                    f"Generate a daily briefing covering these topics:\n{topics_str}\n\n"
                    "Today's briefing:"
                ),
            },
        ],
    )
    return response["message"]["content"]


class DailyBriefingScheduler:
    def __init__(self, log_path: str = "/tmp/day027_run_log.json"):
        self.log_path = log_path
        self.jobs: list[dict] = []

    def add_job(self, name: str, fn, interval_minutes: int) -> None:
        self.jobs.append(build_job(name, fn, interval_minutes))

    def _last_run(self, name: str) -> str | None:
        log = load_run_log(self.log_path)
        runs = [r for r in log if r["name"] == name]
        return runs[-1]["ran_at"] if runs else None

    def due_jobs(self, now: datetime | None = None) -> list[str]:
        return [
            j["name"]
            for j in self.jobs
            if j["enabled"] and is_due(self._last_run(j["name"]), j["interval_minutes"], now)
        ]

    def save_briefing(self, content: str, output_dir: str) -> str:
        today = datetime.now().strftime("%Y-%m-%d")
        path = str(Path(output_dir) / f"briefing_{today}.txt")
        Path(path).write_text(content, encoding="utf-8")
        return path

    def run(
        self,
        topics: list[str],
        output_dir: str,
        model: str = "llama3.2",
        now: datetime | None = None,
    ) -> dict:
        due = self.due_jobs(now)
        content = ai_daily_briefing(topics, model=model)
        path = self.save_briefing(content, output_dir)
        log = load_run_log(self.log_path)
        for name in due:
            log = record_run(log, name, f"briefing saved to {path}")
        save_run_log(self.log_path, log)
        return {"content": content, "path": path, "ran_jobs": due}

## Action 1 — Set Up Scheduler and Jobs

In [ ]:
def _briefing_fn(): pass  # placeholder callable for job config

scheduler = DailyBriefingScheduler(
    log_path=os.path.join(tempfile.gettempdir(), 'day027_run_log.json'),
)
scheduler.add_job('daily_briefing', _briefing_fn, interval_minutes=1440)
print(f'Jobs registered: {[j["name"] for j in scheduler.jobs]}')
print(f'Due jobs (never ran before): {scheduler.due_jobs()}')

## Action 2 — Run the Briefing Pipeline

In [ ]:
TOPICS = [
    'AI Engineering learning progress',
    'Project status update',
]

result = scheduler.run(TOPICS, tempfile.gettempdir())
print(f"Briefing saved: {result['path']}")
print(f"Jobs that ran:  {result['ran_jobs']}")

## Action 3 — Verify Output and Run Log

In [ ]:
# Preview the briefing
print('Briefing preview (first 200 chars):')
print(result['content'][:200])

# Verify run log was updated
log = load_run_log(scheduler.log_path)
print(f'\nRun log has {len(log)} record(s)')
if log:
    last = log[-1]
    print(f"Last entry: name={last['name']!r} status={last['status']!r}")

# Idempotency check: run again — same job should NOT be due
still_due = scheduler.due_jobs()
print(f'\nDue jobs after first run: {still_due} (should be [])')
print('\nScheduling complete!')